## EV County-Level Log Transformation and Normalization

This notebook applies log transformation and feature normalization to the cleaned county-level EV dataset (output csv files saved from EV_project_cleand.ipynb).

Goals:
- reduce heavy right-skew in count-based features using `np.log1p()`
- standardize numeric features for modeling
- save transformed outputs for PCA, clustering, and other unsupervised leadning tasks

In [10]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [11]:
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

In [12]:
# Set project working directory
os.chdir(r"C:\\Users\\alexm\\Documents\\GitHub\\SML_group_project")
print("Working directory:", os.getcwd())

Working directory: C:\Users\alexm\Documents\GitHub\SML_group_project


## 1. Load Data

In [13]:
# Load the cleaned county-level dataset.
county_df = pd.read_csv("data/clean_county_data.csv")

print("\nDataset shape:", county_df.shape)
print("\nColumns:")
print(county_df.columns.tolist())
print("\nData types:")
print(county_df.dtypes)
print("\nFirst 10 rows:")
display(county_df.head(10))


Dataset shape: (39, 17)

Columns:
['county', 'total_EV_count', 'BEV_count', 'PHEV_count', 'avg_model_year', 'avg_electric_range', 'avg_vehicle_age', 'percent_BEV', 'percent_PHEV', 'BEV_to_PHEV_ratio', 'median_income_2025', 'total_chargers', 'level1_chargers', 'level2_chargers', 'dc_fast_chargers', 'fast_charger_share', 'chargers_per_1000_EV']

Data types:
county                   object
total_EV_count            int64
BEV_count                 int64
PHEV_count                int64
avg_model_year          float64
avg_electric_range      float64
avg_vehicle_age         float64
percent_BEV             float64
percent_PHEV            float64
BEV_to_PHEV_ratio       float64
median_income_2025      float64
total_chargers          float64
level1_chargers         float64
level2_chargers         float64
dc_fast_chargers        float64
fast_charger_share      float64
chargers_per_1000_EV    float64
dtype: object

First 10 rows:


,county,total_EV_count,BEV_count,PHEV_count,avg_model_year,avg_electric_range,avg_vehicle_age,percent_BEV,percent_PHEV,BEV_to_PHEV_ratio,median_income_2025,total_chargers,level1_chargers,level2_chargers,dc_fast_chargers,fast_charger_share,chargers_per_1000_EV
0,ADAMS,620,435,185,2020.061290,111.452722,5.938710,0.701613,0.298387,2.351351,69332.46,9.0,0.0,0.0,9.0,1.000000,14.516129
1,ASOTIN,710,452,258,2019.712676,95.222698,6.287324,0.636620,0.363380,1.751938,75647.31,2.0,0.0,2.0,0.0,0.000000,2.816901
2,BENTON,22639,15493,7146,2019.935200,108.205155,6.064800,0.684350,0.315650,2.168066,95571.93,45.0,0.0,30.0,15.0,0.333333,1.987720
3,CHELAN,10014,7883,2131,2020.235171,125.745458,5.764829,0.787198,0.212802,3.699202,84324.53,80.0,12.0,53.0,27.0,0.337500,7.988816
4,CLALLAM,10602,7089,3513,2019.356725,98.694111,6.643275,0.668647,0.331353,2.017933,78138.51,68.0,0.0,45.0,23.0,0.338235,6.413884
5,CLARK,101361,73598,27763,2020.142599,106.294224,5.857401,0.726098,0.273902,2.650938,105087.29,193.0,0.0,132.0,61.0,0.316062,1.904085
6,COLUMBIA,154,121,33,2019.987013,134.473684,6.012987,0.785714,0.214286,3.666667,70875.79,8.0,0.0,7.0,1.0,0.125000,51.948052
7,COWLITZ,8819,6175,2644,2020.086858,107.818630,5.913142,0.700193,0.299807,2.335477,80987.51,28.0,0.0,9.0,19.0,0.678571,3.174963
8,DOUGLAS,3635,2712,899,2019.928473,113.734704,6.071527,0.746080,0.247318,3.016685,79146.07,7.0,0.0,6.0,1.0,0.142857,1.925722
9,FERRY,310,238,72,2020.151613,141.765027,5.848387,0.767742,0.232258,3.305556,62790.88,0.0,0.0,0.0,0.0,0.000000,0.000000


## 2. Identify Numeric Columns & Skewness of Our Dataset

In [14]:
# Detect county column (case insensitive)
county_col_candidates = [col for col in county_df.columns if col.lower() == "county"]
if not county_col_candidates:
    raise ValueError("Could not find a county column in the dataset.")
county_col = county_col_candidates[0]

# Identify numeric columns for analysis
# if data cleaning was done correctly, all columns should be numeric except for the county column
numeric_cols = [
    col for col in county_df.select_dtypes(include=[np.number]).columns
    if col != county_col
]

print("County column:", county_col)
print("\nNumeric columns:")
print(numeric_cols)

print("\nSummary statistics for numeric columns:")
display(county_df[numeric_cols].describe().T)

# Calculate skewness for numeric columns
skew_table = county_df[numeric_cols].skew(numeric_only=True).to_frame("skewness")
skew_table = skew_table.sort_values("skewness", key=lambda s: s.abs(), ascending=False)

print("\nSorted skewness table (highest absolute skew first):")
display(skew_table)

County column: county

Numeric columns:
['total_EV_count', 'BEV_count', 'PHEV_count', 'avg_model_year', 'avg_electric_range', 'avg_vehicle_age', 'percent_BEV', 'percent_PHEV', 'BEV_to_PHEV_ratio', 'median_income_2025', 'total_chargers', 'level1_chargers', 'level2_chargers', 'dc_fast_chargers', 'fast_charger_share', 'chargers_per_1000_EV']

Summary statistics for numeric columns:


,count,mean,std,min,25%,50%,75%,max
total_EV_count,39.0,44397.487179,145873.829414,34.000000,2100.000000,6469.000000,19038.500000,894695.000000
BEV_count,39.0,34171.076923,116333.511156,11.000000,1273.500000,4408.000000,13667.500000,713944.000000
PHEV_count,39.0,10225.307692,29630.964809,23.000000,809.000000,2038.000000,5371.000000,180748.000000
avg_model_year,39.0,2019.882930,0.514577,2018.000000,2019.706917,2019.995672,2020.162038,2020.677193
avg_electric_range,39.0,106.786601,14.701700,62.566667,99.941830,105.823985,113.543127,141.765027
avg_vehicle_age,39.0,6.117070,0.514577,5.322807,5.837962,6.004328,6.293083,8.000000
percent_BEV,39.0,0.687231,0.087216,0.323529,0.660433,0.700193,0.733781,0.806135
percent_PHEV,39.0,0.312563,0.087304,0.193865,0.266212,0.299807,0.339567,0.676471
BEV_to_PHEV_ratio,39.0,2.395558,0.789489,0.478261,1.945706,2.335477,2.756440,4.158220
median_income_2025,39.0,83781.304103,15515.654597,60663.480000,72026.225000,81477.360000,92797.145000,126124.370000



Sorted skewness table (highest absolute skew first):


,skewness
level1_chargers,6.183787
level2_chargers,5.974053
total_chargers,5.901915
BEV_count,5.565499
total_EV_count,5.523755
PHEV_count,5.319469
dc_fast_chargers,4.820511
chargers_per_1000_EV,4.440001
percent_BEV,-2.063757
percent_PHEV,2.060417


The skewness analysis above shows that all EV and charger count variables exhibit extreme right‑skew (skewness between 4 and 6), driven by a few large counties such as King and Snohomish. These variables span several orders of magnitude and contain large outliers, making them unsuitable for PCA or clustering in their raw form.

Therefore, we apply a log1p transformation to all count‑based features to compress outliers and reduce skew: `total_EV_count`, `BEV_count`, `PHEV_count`, `total_chargers`, `level1_chargers`, `level2_chargers`, `dc_fast_chargers`, `chargers_per_1000_EV`


In contrast, variables such as percentages, ratios, average model year, average vehicle age, electric range, and median income exhibit mild skew or are bounded between 0 and 1. These are kept on their original scales because they are ratios/percentages/averages where log transformation can reduce interpretability or distort meaning: `percent_BEV`, `percent_PHEV`, `BEV_to_PHEV_ratio`, `avg_model_year`, `avg_vehicle_age`, `avg_electric_range`, `median_income_2025`

## 3. Log Transform Skewed Features

In [15]:
# We log-transform heavy count features to compress large outliers and reduce right-skew
log_transform_cols = [
    "total_EV_count",
    "BEV_count",
    "PHEV_count",
    "total_chargers",
    "level1_chargers",
    "level2_chargers",
    "dc_fast_chargers",
    "chargers_per_1000_EV",
]

# These are kept on their original scales because they are ratios/percentages/averages 
not_log_transform_cols = [
    "percent_BEV",
    "percent_PHEV",
    "BEV_to_PHEV_ratio",
    "avg_model_year",
    "avg_vehicle_age",
    "avg_electric_range",
    "median_income_2025",
]

# Check if all required columns are present
missing_for_log = [c for c in log_transform_cols if c not in county_df.columns]
if missing_for_log:
    raise ValueError(f"Missing required columns for log transform: {missing_for_log}")

# Create a copy of the dataframe to avoid modifying the original
county_log = county_df.copy()
for col in log_transform_cols:          # apply log transformation
    county_log[f"{col}_log"] = np.log1p(county_log[col])

print("Created log-transformed columns:")
print([f"{c}_log" for c in log_transform_cols])
print("\nColumns intentionally not log-transformed:")
print(not_log_transform_cols)

Created log-transformed columns:
['total_EV_count_log', 'BEV_count_log', 'PHEV_count_log', 'total_chargers_log', 'level1_chargers_log', 'level2_chargers_log', 'dc_fast_chargers_log', 'chargers_per_1000_EV_log']

Columns intentionally not log-transformed:
['percent_BEV', 'percent_PHEV', 'BEV_to_PHEV_ratio', 'avg_model_year', 'avg_vehicle_age', 'avg_electric_range', 'median_income_2025']


In [16]:
# Table: Compare original vs log-transformed skewness
skew_compare_rows = []

for col in log_transform_cols:
    log_col = f"{col}_log"
    before_skew = county_log[col].skew()
    after_skew = county_log[log_col].skew()

    skew_compare_rows.append({
        "feature": col,
        "skew_before": before_skew,
        "skew_after": after_skew,
    })

skew_compare = pd.DataFrame(skew_compare_rows).sort_values(
    by="skew_before", key=lambda s: s.abs(), ascending=False
)

print("Skewness comparison before vs after log transform:")
display(skew_compare)


Skewness comparison before vs after log transform:


,feature,skew_before,skew_after
4,level1_chargers,6.183787,3.862907
5,level2_chargers,5.974053,0.325946
3,total_chargers,5.901915,0.160641
1,BEV_count,5.565499,-0.141700
0,total_EV_count,5.523755,-0.047376
2,PHEV_count,5.319469,-0.077960
6,dc_fast_chargers,4.820511,0.217964
7,chargers_per_1000_EV,4.440001,0.585049


Log transformation was applied to the highly skewed count‑based EV and charger features to reduce extreme right‑skewness and make the data suitable for PCA and clustering. Whereas percentage, ratio, and average features were left untransformed because they are already well‑behaved.

**What we can infer from the original vs log-transformed sknewness table?**

1. **Before log transformation**, all EV and charger count variables had extreme right skew (skewness between 4.4 and 6.2).  
   This means:
   - A few counties (like King) have massively higher counts.
   - Most counties have very small values.
   - Distributions have long right tails.
   - PCA and clustering would be dominated by these outliers.

2. **After log transformation**, the skewness dropped dramatically for almost all variables:
   - Many features moved from ~5–6 skew to near 0.
   - Examples:
     - level2_chargers: 5.97 to 0.33
     - total_chargers: 5.90 to 0.16
     - BEV_count: 5.56 to –0.14

   This means:
   - Outliers were compressed.
   - Distributions became more symmetric.
   - Variance is stabilized.
   - Features are now suitable for PCA and clustering.

3. **One exception: level1_chargers**
   - Skew dropped from 6.18 → 3.86, still high.
   - Likely because:
     - Many counties have zero Level 1 chargers.
     - A few have 100+, so even log can’t fully fix it.

4. **Why these variables were chosen for log transform**
   - They are counts with huge magnitude differences.
   - They have the highest skew in the dataset.
   - They would dominate PCA/clustering if left untransformed.

5. **Why other variables were *not* log transformed**
   - They are percentages, ratios, averages, or income.
   - They have mild or moderate skew, not extreme.
   - Log transform would:
     - distort interpretation,
     - break the meaning of percentages,
     - or provide no benefit.


## 4. Standardize Features

StandardScaler is important for PCA and clustering because these methods are distance/variance based. Without scaling, large-magnitude features dominate the model and bias the results.

**Steps Performed During Scaling:**

**1. Drop the county column**
- The county name is categorical and should not be scaled.
- Only numeric features (including log‑transformed ones) are kept for scaling.

**2. Fit the StandardScaler and transform the numeric data**
- Each feature is standardized to:
  - mean = **0**
  - standard deviation = **1**
- This removes magnitude differences between variables.

**3. Create a new scaled DataFrame**
- The scaled NumPy array is converted back into a DataFrame.
- Column names and row order are preserved for consistency.

**4. Add the county column back**
- County names are reinserted as the first column.
- This keeps the dataset interpretable and easy to merge with other outputs.

**5. Check summary statistics**
- After scaling, all numeric features should have:
  - mean ≈ **0**
  - std ≈ **1**
- This confirms that scaling worked correctly.

**6. Preview the first few rows**
- Ensures the dataset looks correct:
  - county names intact  
  - numeric values standardized  
  - no missing or misaligned columns

In [17]:
# Drop the county column since it's not a numeric feature (i.e. it's categorical)
county_numeric_matrix = county_log.drop(columns=[county_col]).select_dtypes(include=[np.number])

# Fit scaler and transform all numeric features (including log-transformed features)
scaler = StandardScaler()
scaled_values = scaler.fit_transform(county_numeric_matrix)

# Create scaled dataframe
county_scaled = pd.DataFrame(
    scaled_values,
    columns=county_numeric_matrix.columns,
    index=county_log.index,
)

# Add the county back as first column
county_scaled.insert(0, county_col, county_log[county_col].values)

# Show summary stats of scaled numeric features.
print("Scaled dataset shape:", county_scaled.shape)
print("\nSummary statistics of scaled numeric features:")
display(county_scaled.drop(columns=[county_col]).describe().T)
print("\nFirst 5 rows of scaled dataset:")
display(county_scaled.head())

Scaled dataset shape: (39, 25)

Summary statistics of scaled numeric features:


,count,mean,std,min,25%,50%,75%,max
total_EV_count,39.0,2.490885e-17,1.013072,-0.308098,-0.293750,-0.263408,-0.176114,5.905192
BEV_count,39.0,2.917894e-17,1.013072,-0.297478,-0.286483,-0.259187,-0.178552,5.919698
PHEV_count,39.0,2.206212e-17,1.013072,-0.348813,-0.321940,-0.279921,-0.165967,5.830112
avg_model_year,39.0,-6.542430e-13,1.013072,-3.707016,-0.346525,0.221960,0.549494,1.563704
avg_electric_range,39.0,-1.634376e-15,1.013072,-3.047131,-0.471663,-0.066332,0.465582,2.410312
avg_vehicle_age,39.0,1.392049e-15,1.013072,-1.563704,-0.549494,-0.221960,0.346525,3.707016
percent_BEV,39.0,9.251859e-18,1.013072,-4.224637,-0.311276,0.150558,0.540703,1.381144
percent_PHEV,39.0,3.124281e-16,1.013072,-1.377368,-0.537852,-0.148015,0.313357,4.222797
BEV_to_PHEV_ratio,39.0,4.469359e-16,1.013072,-2.460278,-0.577251,-0.077097,0.463084,2.261849
median_income_2025,39.0,-7.572290e-16,1.013072,-1.509445,-0.767531,-0.150433,0.588676,2.764730



First 5 rows of scaled dataset:


,county,total_EV_count,BEV_count,PHEV_count,avg_model_year,avg_electric_range,avg_vehicle_age,percent_BEV,percent_PHEV,BEV_to_PHEV_ratio,median_income_2025,total_chargers,level1_chargers,level2_chargers,dc_fast_chargers,fast_charger_share,chargers_per_1000_EV,total_EV_count_log,BEV_count_log,PHEV_count_log,total_chargers_log,level1_chargers_log,level2_chargers_log,dc_fast_chargers_log,chargers_per_1000_EV_log
0,ADAMS,-0.304028,-0.293785,-0.343275,0.351146,0.321535,-0.351146,0.167054,-0.164494,-0.056726,-0.943417,-0.284265,-0.185346,-0.283009,-0.281631,2.701390,1.016734,-1.114216,-1.056557,-1.189832,-0.650199,-0.354030,-1.817486,0.251486,1.534355
1,ASOTIN,-0.303403,-0.293637,-0.340779,-0.335187,-0.796852,0.335187,-0.587885,0.589688,-0.825894,-0.531097,-0.302321,-0.185346,-0.277127,-0.463012,-0.849210,-0.369807,-1.047830,-1.038530,-1.017391,-1.403685,-0.354030,-1.146074,-1.099162,-0.357901
2,BENTON,-0.151110,-0.162655,-0.105280,0.102907,0.097750,-0.102907,-0.033466,0.035825,-0.291918,0.769852,-0.191406,-0.185346,-0.194781,-0.160711,0.334323,-0.468077,0.649723,0.626235,0.710560,0.304856,-0.354030,0.281179,0.527180,-0.688372
3,CHELAN,-0.238788,-0.228926,-0.276742,0.693473,1.306427,-0.693473,1.161181,-1.157625,1.672837,0.035469,-0.101126,0.283529,-0.127139,0.081131,0.349117,0.243146,0.249647,0.307819,0.080535,0.658957,2.363072,0.620363,0.855439,0.797796
4,CLALLAM,-0.234705,-0.235840,-0.229491,-1.035965,-0.557642,1.035965,-0.215862,0.218038,-0.484569,-0.368438,-0.132080,-0.185346,-0.150667,0.000517,0.351728,0.056492,0.277632,0.257791,0.340796,0.558610,-0.354030,0.522370,0.765017,0.537893


## What We Can Infer From the Scaled Dataset

**1. Scaling behaved correctly**
- All numeric features have mean ≈ 0 and std ≈ 1, confirming StandardScaler applied consistently across all variables.

**2. True structural outliers remain visible**
- Several EV and charger count features show values around +5 to +6 SD.
- These are not scaling artifacts; they reflect genuinely extreme counties with unusually high EV adoption or infrastructure.

**3. Log-transformed features show reduced extremeness**
- Log versions of EV and charger counts have much smaller standardized ranges (max ~2–3 SD).
- This confirms the log transform successfully compressed heavy right‑skew and reduced dominance of large counties.

**4. Percentage-based features capture meaningful behavioral variation**
- `percent_BEV` and `percent_PHEV` span wide standardized ranges, indicating strong differences in BEV/PHEV preference across counties.
- These features will likely contribute to distinct PCA components.

**5. The dataset is now suitable for PCA and clustering**
- Variance is balanced across features.
- No single variable’s magnitude will overwhelm the analysis.
- Both raw and log-transformed versions coexist cleanly, enabling PCA to separate “scale effects” from “behavioral patterns.”

## 5. Save Outputs Files

In [18]:
log_output_path = "data/county_log_transformed.csv"
scaled_output_path = "data/county_scaled.csv"

county_log.to_csv(log_output_path, index=False)
county_scaled.to_csv(scaled_output_path, index=False)

print(f"Saved log-transformed dataset to: {log_output_path}")
print(f"Saved scaled dataset to: {scaled_output_path}")

Saved log-transformed dataset to: data/county_log_transformed.csv
Saved scaled dataset to: data/county_scaled.csv
